# LogP prediction with Gradient Boosting

**Project question:** Can molecular descriptors predict lipophilicity (LogP), and can machine learning help predict bioactivity for drug design?

This notebook completes the first, quantitative part of the project: predicting **Mannhold LogP** from molecular descriptors and comparing predictions with the dataset's **XLogP** column.

## Important scientific choices

- `Mannhold LogP` is the modelling target.
- `XLogP` is a comparator, **not a model input**. Including it as a feature would make the comparison circular.
- `index`, `row ID`, molecule names, formula and SMILES are identifiers/text, not numeric model inputs.
- Repeated molecular structures are kept in the same data split.
- A scaffold-based split is also used because it better tests generalisation to unfamiliar chemical frameworks.
- The dataset contains calculated LogP values. The experiment therefore measures agreement with a calculated reference, not accuracy against experimental partition-coefficient measurements.
- Extreme observations are investigated and retained in the primary analysis unless a documented data error is found.

## 1. Setup and load the dataset

Place `properties.csv` in the same folder as this notebook. If it is elsewhere, change `DATA_PATH`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("properties.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find properties.csv. Put it beside this notebook or update DATA_PATH."
    )

raw = pd.read_csv(DATA_PATH)
print("Dataset shape:", raw.shape)
display(raw.head())

## 2. Inspect and clean the data

The CSV stores some numeric descriptors as text. Convert numeric-looking columns to numbers and turn entries such as `#NUM!` into missing values. We do not impute here: imputation is fitted inside the training pipeline later, preventing data leakage.

In [ ]:
df = raw.copy()

# Convert every column except known text/identifier columns where possible.
text_columns = {"row ID", "Molecule", "Molecule name", "Molecular Formula"}
for col in df.columns:
    if col not in text_columns:
        df[col] = pd.to_numeric(df[col].replace(["#NUM!", "NUM!", ""], np.nan),
                                errors="coerce")

print("Rows and columns:", df.shape)
print("\nColumn types:")
display(df.dtypes.astype(str).to_frame("dtype"))

print("\nMissing values (largest first):")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))

print("\nExact duplicate rows:", df.duplicated().sum())
print("Repeated SMILES / molecule strings:", df["Molecule"].duplicated().sum())
print("Unique molecule strings:", df["Molecule"].nunique())

## 3. Explore the target and descriptors

The distribution has a long upper tail. Report the full distribution and investigate extreme values rather than removing them automatically.

In [ ]:
TARGET = "Mannhold LogP"
COMPARATOR = "XLogP"

display(df[[TARGET, COMPARATOR]].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET].dropna(), bins=60)
axes[0].set_title("Mannhold LogP distribution")
axes[0].set_xlabel("Mannhold LogP")
axes[0].set_ylabel("Number of compounds")

axes[1].hist(df[COMPARATOR].dropna(), bins=60)
axes[1].set_title("XLogP distribution")
axes[1].set_xlabel("XLogP")
axes[1].set_ylabel("Number of compounds")
plt.tight_layout()
plt.show()

print("Highest Mannhold LogP records:")
display(df.nlargest(10, TARGET)[
    ["Molecule name", TARGET, COMPARATOR, "Molecular Weight",
     "Topological Polar Surface Area", "Molecule"]
])

## 4. Correlation analysis

Correlation is useful for initial exploration, but it does not establish causation. Highly correlated descriptors can contain overlapping information.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[numeric_cols].corr(numeric_only=True)

target_corr = corr[TARGET].drop(TARGET).sort_values()
print("Strongest negative correlations with Mannhold LogP:")
display(target_corr.head(10).to_frame("correlation"))
print("Strongest positive correlations with Mannhold LogP:")
display(target_corr.tail(10).sort_values(ascending=False).to_frame("correlation"))

plt.figure(figsize=(9, 7))
target_corr.sort_values().plot(kind="barh")
plt.title("Correlation of numeric descriptors with Mannhold LogP")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## 5. Define features and prevent leakage

Use numeric molecular descriptors only. Exclude:
- the target `Mannhold LogP`;
- the comparator `XLogP`;
- row/index identifiers;
- textual identifiers.

This keeps the model from using another LogP estimate to predict the target. The features are descriptors of the molecule, not experimental bioactivity measurements.

In [ ]:
exclude = {
    TARGET, COMPARATOR, "index",
    "row ID", "Molecule", "Molecule name", "Molecular Formula"
}

feature_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in exclude
]

model_data = df.loc[df[TARGET].notna()].copy()
X = model_data[feature_cols]
y = model_data[TARGET].astype(float)

# Group identical molecule strings together so duplicates cannot appear in both sets.
groups_molecule = model_data["Molecule"].fillna(
    model_data.index.astype(str)
).astype(str)

print("Number of rows used:", len(model_data))
print("Number of numeric features:", len(feature_cols))
print("Features:", feature_cols)

## 6. Create a molecule-grouped train/test split

A random row split can put duplicate structures in both training and test data. GroupShuffleSplit prevents identical molecule strings from crossing the split.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups_molecule))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Molecule-string overlap:",
      len(set(groups_molecule.iloc[train_idx]) & set(groups_molecule.iloc[test_idx])))

## 7. Train baseline and Gradient Boosting models

The median imputer is part of each pipeline, so its values are learned from training data only. Gradient Boosting does not require feature scaling.

In [ ]:
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=2,
        loss="huber", random_state=RANDOM_STATE
    )
}

fitted = {}
predictions = {}
rows = []

for name, estimator in models.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    predictions[name] = pred
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })

results = pd.DataFrame(rows).sort_values("MAE")
display(results)

## 8. Compare Gradient Boosting with XLogP

XLogP is evaluated on the **same test compounds** as the machine-learning models. It is not used to train Gradient Boosting.

Interpretation: this comparison measures how closely each estimate agrees with the dataset's Mannhold LogP column. It does not prove which method is closer to experimental LogP.

In [ ]:
gb_pred = predictions["Gradient boosting"]
xlogp_test = model_data[COMPARATOR].iloc[test_idx]

comparison = pd.DataFrame([
    {
        "Method": "Gradient Boosting",
        "MAE": mean_absolute_error(y_test, gb_pred),
        "RMSE": mean_squared_error(y_test, gb_pred) ** 0.5,
        "R2": r2_score(y_test, gb_pred)
    },
    {
        "Method": "XLogP",
        "MAE": mean_absolute_error(y_test, xlogp_test),
        "RMSE": mean_squared_error(y_test, xlogp_test) ** 0.5,
        "R2": r2_score(y_test, xlogp_test)
    }
])
display(comparison)

## 9. Inspect predictions and residuals

A good model should show predictions close to the diagonal in the observed-versus-predicted plot. Residual plots can reveal systematic errors, such as under-predicting very high LogP compounds.

In [ ]:
residuals = y_test.to_numpy() - gb_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, gb_pred, alpha=0.35)
lo = min(y_test.min(), gb_pred.min())
hi = max(y_test.max(), gb_pred.max())
axes[0].plot([lo, hi], [lo, hi], linestyle="--")
axes[0].set_xlabel("Observed Mannhold LogP")
axes[0].set_ylabel("Predicted Mannhold LogP")
axes[0].set_title("Observed vs predicted")

axes[1].scatter(gb_pred, residuals, alpha=0.35)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted Mannhold LogP")
axes[1].set_ylabel("Residual (observed - predicted)")
axes[1].set_title("Residual plot")
plt.tight_layout()
plt.show()

print("Median absolute error:",
      np.median(np.abs(residuals)))

## 10. Investigate extreme target values (sensitivity analysis)

Do not delete extreme observations merely to improve the scores. Report full-test metrics as the primary result. A secondary analysis on the central target range can show how much extreme values affect RMSE.

The threshold below is an illustrative diagnostic, not a claim that values outside it are invalid.

In [ ]:
central_mask = y_test.between(-5, 10)
if central_mask.sum() > 0:
    print("Central-range test rows:", int(central_mask.sum()), "of", len(y_test))
    print("Full test MAE:", mean_absolute_error(y_test, gb_pred))
    print("Central-range MAE:",
          mean_absolute_error(y_test[central_mask], gb_pred[central_mask]))
    print("Full test RMSE:", mean_squared_error(y_test, gb_pred) ** 0.5)
    print("Central-range RMSE:",
          mean_squared_error(y_test[central_mask], gb_pred[central_mask]) ** 0.5)
else:
    print("No test compounds in the selected central range.")

## 11. Explain which features matter

Permutation importance measures how much test performance worsens when one feature is shuffled. It is a model-specific measure of predictive usefulness, not proof that a feature causes lipophilicity.

The computation is performed on a sample to keep runtime manageable.

In [ ]:
sample_n = min(2000, len(X_test))
rng = np.random.default_rng(RANDOM_STATE)
sample_positions = rng.choice(len(X_test), size=sample_n, replace=False)
X_imp = X_test.iloc[sample_positions]
y_imp = y_test.iloc[sample_positions]

perm = permutation_importance(
    fitted["Gradient boosting"], X_imp, y_imp,
    n_repeats=5, random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error", n_jobs=-1
)

importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance (MAE increase)": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance (MAE increase)", ascending=False)

display(importance.head(15))

top = importance.head(15).sort_values("Importance (MAE increase)")
plt.figure(figsize=(9, 6))
plt.barh(top["Feature"], top["Importance (MAE increase)"],
         xerr=top["Std"])
plt.xlabel("Increase in MAE after shuffling")
plt.title("Gradient Boosting permutation feature importance")
plt.tight_layout()
plt.show()

## 12. Stronger generalisation test: molecular scaffold split

A scaffold split asks whether the model can predict compounds whose core chemical frameworks are absent from training. This is a more demanding test for drug-discovery applications.

RDKit is used to calculate Bemis–Murcko scaffolds from SMILES. If RDKit is unavailable or a molecule cannot be parsed, this section will stop with an explanatory message. Acyclic molecules with no Murcko scaffold are grouped by canonical SMILES to avoid putting duplicate structures in separate groups.

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem.Scaffolds import MurckoScaffold

    def scaffold_group(smiles):
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return "UNPARSEABLE:" + str(smiles)
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        scaffold_smiles = Chem.MolToSmiles(scaffold)
        if scaffold_smiles == "":
            return "ACYCLIC:" + Chem.MolToSmiles(mol)
        return scaffold_smiles

    scaffold_groups = model_data["Molecule"].map(scaffold_group)
    scaffold_splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.20, random_state=RANDOM_STATE
    )
    scaffold_train, scaffold_test = next(
        scaffold_splitter.split(X, y, groups=scaffold_groups)
    )

    scaffold_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=2,
            loss="huber",
            random_state=RANDOM_STATE
        ))
    ])
    scaffold_model.fit(X.iloc[scaffold_train], y.iloc[scaffold_train])
    scaffold_pred = scaffold_model.predict(X.iloc[scaffold_test])
    y_scaffold_test = y.iloc[scaffold_test]

    print("Scaffold train/test sizes:",
          len(scaffold_train), len(scaffold_test))
    print("Scaffold overlap:",
          len(set(scaffold_groups.iloc[scaffold_train]) &
              set(scaffold_groups.iloc[scaffold_test])))
    print("Scaffold-split MAE:",
          mean_absolute_error(y_scaffold_test, scaffold_pred))
    print("Scaffold-split RMSE:",
          mean_squared_error(y_scaffold_test, scaffold_pred) ** 0.5)
    print("Scaffold-split R2:",
          r2_score(y_scaffold_test, scaffold_pred))

except ImportError:
    print("RDKit is not installed. Install it with: conda install -c conda-forge rdkit")

## 13. Optional hyperparameter tuning

Run this after the initial results. It uses grouped cross-validation so duplicate molecule strings stay together within each fold. Tuning should be done using training data only; the test set remains untouched until final evaluation.

This search can take several minutes.

In [ ]:
# Optional: set RUN_TUNING = True when ready.
RUN_TUNING = False

if RUN_TUNING:
    search_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE))
    ])

    param_distributions = {
        "model__n_estimators": [150, 250, 350, 450],
        "model__learning_rate": [0.03, 0.05, 0.08, 0.1],
        "model__max_depth": [1, 2, 3],
        "model__min_samples_leaf": [2, 3, 5, 10],
        "model__loss": ["squared_error", "huber"]
    }

    cv = GroupKFold(n_splits=3)
    search = RandomizedSearchCV(
        search_pipe,
        param_distributions=param_distributions,
        n_iter=12,
        scoring="neg_mean_absolute_error",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train, y_train, groups=groups_molecule.iloc[train_idx])

    print("Best parameters:", search.best_params_)
    print("Best grouped-CV MAE:", -search.best_score_)

    tuned_pred = search.best_estimator_.predict(X_test)
    print("Held-out test MAE:", mean_absolute_error(y_test, tuned_pred))
    print("Held-out test RMSE:", mean_squared_error(y_test, tuned_pred) ** 0.5)
    print("Held-out test R2:", r2_score(y_test, tuned_pred))

## 14. Bioactivity: what the second dataset must contain

The properties dataset cannot train a target-specific bioactivity model because it has no biological target or assay activity labels.

For the next stage, obtain a ChEMBL export for **one selected protein target**. Ideally include:
- canonical SMILES or another chemical structure identifier;
- target name and target identifier;
- assay type and assay confidence;
- standard type (for example IC50 or Ki);
- standard value and standard units;
- activity relation (prefer exact `=`);
- assay organism and assay description.

Do not mix IC50, Ki, EC50 and other endpoints as though they were identical. For a first model, select one endpoint and one target, filter to comparable assays, standardise units, remove ambiguous measurements and handle duplicate compounds carefully.

For an IC50 regression task, convert nM to molar units and calculate:

`pIC50 = -log10(IC50 in mol/L)`

For a classification task, define an active/inactive threshold before modelling and justify it from the literature and assay context. Exclude borderline or ambiguous measurements rather than assigning arbitrary labels.

A possible follow-on workflow:
1. Download a target-specific activity table from ChEMBL.
2. Standardise structures and remove invalid SMILES.
3. Calculate molecular descriptors from SMILES (RDKit).
4. Join the descriptors to activity labels using a stable compound identifier or canonical structure.
5. Train a separate model to predict pIC50 or active/inactive status.
6. Use a scaffold split and report MAE/RMSE/R² for regression or ROC-AUC/PR-AUC, sensitivity and specificity for classification.

**Important:** LogP can be one of the descriptors in the bioactivity model, but it should not be treated as a substitute for biological assay data. Bioactivity also depends on target binding, molecular shape, charge, stereochemistry, solubility, permeability and assay conditions.

## 15. Suggested conclusion structure

When the notebook has been run, write the conclusion using the actual final metrics:

1. State which descriptors were most useful to predict Mannhold LogP.
2. Report Gradient Boosting MAE, RMSE and R² on the held-out test set.
3. Compare those values with XLogP on exactly the same compounds.
4. Compare the random molecule-grouped split with the scaffold split. A drop in performance on the scaffold split indicates weaker generalisation to new chemical frameworks.
5. Explain the effect of extreme observations.
6. State the limitation that Mannhold LogP is a calculated reference rather than experimental ground truth.
7. Explain that a separate target-specific bioactivity dataset is required before making claims about predicting biological activity.

Do not claim that the model is better than XLogP for experimental LogP unless you validate both against experimental measurements.